# wcd_pipeline — Full Pipeline (Discovery → Validation → Catalog)
Part of the Public Webcam Discovery System.

## Pipeline execution model

The pipeline uses a **streaming parallel producer-consumer architecture** rather than running agents in sequence:

```
DirectoryAgent.stream()  ─┐
                          ├─► asyncio.Queue ──► ValidationAgent.run_from_queue()
SearchAgent.stream()     ─┘
```

1. **DirectoryAgent** and **SearchAgent** run **concurrently** — both start immediately and stream `CameraCandidate` objects into a shared bounded queue as they are found (batch-by-batch for directory crawling, city-by-city for search)
2. **ValidationAgent** starts processing **immediately** — it drains the queue in batches of 100 so HTTP probing and geocoding begin as soon as the first batch of candidates arrives, without waiting for all discovery to finish
3. **CatalogAgent** runs after all records are collected — deduplication requires the complete record set

### Why this matters
With a 25 s read timeout and up to 50 concurrent probes, waiting for full discovery before validation adds unnecessary serial latency.  The streaming model cuts total pipeline time roughly in half for large runs (e.g. Tier 1: ~40 min → ~20 min).

## Geocoding
Camera location strings are passed to an **Ollama LLM** (`USE_LLM_GEODECODE = True` by default).
The model returns structured `location`, `latitude`, `longitude`, `country`, `region`, and `continent`.
Results are cached process-wide so each unique location is resolved only once per pipeline run.
Set `WCD_USE_LLM_GEODECODE=false` (env var) to revert to the Nominatim fallback chain.

## Notebook vs full pipeline
This notebook runs agents **step-by-step** so you can inspect intermediate outputs at each stage.
To run the complete streaming pipeline with a single command, see **Step 2b** below.

In [ ]:
# Environment setup — run this cell first every session
import subprocess, sys, os
from pathlib import Path

IN_COLAB     = "google.colab" in sys.modules
IN_SAGEMAKER = os.environ.get("SM_TRAINING_ENV") is not None

# ── 1. Locate / clone the project ────────────────────────────────────────────
if IN_COLAB:
    repo_dir = Path("/content/webcam-discovery")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/dshipley71/webcam-discovery", str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
elif IN_SAGEMAKER:
    repo_dir = Path(os.environ.get("SM_MODEL_DIR", "/opt/ml/model")) / "webcam-discovery"
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/dshipley71/webcam-discovery", str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
else:
    # Local: assume CWD is already the project root (where pyproject.toml lives)
    repo_dir = Path.cwd()

print(f"Project root: {repo_dir}")

# ── 2. Install the package using the absolute path ───────────────────────────
# Using repo_dir directly avoids any CWD ambiguity (relative "." can resolve to
# the wrong directory if the kernel was restarted or chdir ran more than once).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{repo_dir}[notebooks]", "-q"],
    check=True,
)

# ── 3. Verify importability — fallback: insert src/ into sys.path ─────────────
# pip install -e works in a fresh kernel but can be lost after a kernel restart
# if the site-packages link is stale.  Inserting src/ ensures the package is
# always resolvable without a second install.
src_path = str(repo_dir / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import webcam_discovery  # noqa: F401  — raises ImportError if src layout is broken
from webcam_discovery.config import settings
settings.ensure_dirs()

print(f"webcam_discovery loaded from: {Path(webcam_discovery.__file__).parent}")
print("Ready ✓")

## Step 1 — Ollama API key configuration

`GeoEnrichmentSkill` uses **Ollama cloud** for geocoding (`USE_LLM_GEODECODE = True` by default).
Supply your API key via **one** of these methods — highest priority wins:

| Method | How |
|--------|-----|
| **Colab secret** | Add a secret named `OLLAMA_API_KEY` in the Colab Secrets panel (🔑) |
| **Manual** | Set `OLLAMA_API_KEY = "sk-..."` in the cell below |
| **Env var** | Set `WCD_OLLAMA_API_KEY=<key>` in your shell or a `.env` file |

Leave `OLLAMA_API_KEY = ""` to connect without authentication (local Ollama or a public endpoint).

In [ ]:
import os

# ── Ollama API key resolution ─────────────────────────────────────────────────
# Priority: Colab userdata secret → manual value below → already-set env var

_ollama_key = ""

# 1. Google Colab userdata secret (recommended for Colab sessions)
try:
    from google.colab import userdata  # type: ignore
    _ollama_key = userdata.get("OLLAMA_API_KEY") or ""
    if _ollama_key:
        print(f"Ollama API key loaded from Colab secrets ✓  (length={len(_ollama_key)})")
except Exception:
    pass

# 2. Manual fallback — paste your key here if not using Colab secrets
if not _ollama_key:
    OLLAMA_API_KEY = ""   # ← paste key here, e.g. "sk-ollama-..."
    _ollama_key = OLLAMA_API_KEY

# 3. Propagate to environment so settings (pydantic-settings) picks it up
if _ollama_key:
    os.environ["WCD_OLLAMA_API_KEY"] = _ollama_key
    if "Colab" not in (globals().get("_source", "") or ""):
        print(f"Ollama API key set from manual value ✓  (length={len(_ollama_key)})")
else:
    print("No Ollama API key set — connecting without authentication")
    print("  (fine for a local Ollama instance; cloud endpoints require a key)")

# ── Optional: override endpoint / model ──────────────────────────────────────
# Uncomment and edit to point at a local Ollama instance or a different model.
# os.environ["WCD_OLLAMA_BASE_URL"] = "http://localhost:11434"  # local Ollama
os.environ["WCD_OLLAMA_BASE_URL"] = "https://ollama.com"  # Ollama cloud (default)
os.environ["WCD_OLLAMA_MODEL"]    = "gemma3:27b"               # model (default)

# ── Confirm active settings ───────────────────────────────────────────────────
# Re-import settings so any env-var overrides above take effect.
from importlib import reload
import webcam_discovery.config as _cfg_mod
reload(_cfg_mod)
from webcam_discovery.config import settings  # noqa: F811 — intentional reload

print()
print(f"Geocoding mode  : {'LLM via Ollama' if settings.use_llm_geodecode else 'Nominatim (legacy)'}")
print(f"Ollama endpoint : {settings.ollama_base_url}")
print(f"Ollama model    : {settings.ollama_model}")
print(f"API key present : {'yes' if settings.ollama_api_key else 'no'}")

## Step 2a - Build environment

Install the package with dev dependencies so the pipeline and tests are available.

In [ ]:
import subprocess, sys
from pathlib import Path

# repo_dir is set by the env setup cell above.
# Fallback to CWD so this cell is also safe to run standalone.
try:
    _repo = repo_dir  # type: ignore[name-defined]
except NameError:
    _repo = Path.cwd()

# Install dev extras (pytest, respx, etc.) using the same absolute-path pattern
# as the env setup cell to avoid CWD ambiguity.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{_repo}[dev]", "-q"],
    check=True,
)
print("Dev dependencies installed ✓")

## Step 2b — Verify the source registry and optionally run the full streaming pipeline

Before launching the pipeline, inspect the checked-in `SOURCES.md` file rather than overwriting it from the notebook.
This keeps the runtime aligned with the repository and avoids accidental drift between notebook content and the canonical source registry.


In [ ]:
from pathlib import Path

sources_path = Path("SOURCES.md")
if not sources_path.exists():
    print("SOURCES.md not found in the project root")
else:
    text = sources_path.read_text(encoding="utf-8")
    lines = text.splitlines()
    print(f"Using checked-in source registry: {sources_path} ({len(lines)} lines)")
    print()
    print("\n".join(lines[:80]))
    if len(lines) > 80:
        print("\n... (truncated preview) ...")


In [ ]:
# Full streaming pipeline — DirectoryAgent + SearchAgent run in parallel and
# feed ValidationAgent via asyncio.Queue so validation overlaps with discovery.
# Equivalent to: wcd-pipeline --tier 1
!python scripts/run_pipeline.py --tier 1

## Step 3a — Run DirectoryAgent (Tier 1)

Crawls all Tier 1 sources from `SOURCES.md` and records **all discovered camera candidate links**.
Direct `.m3u8` HLS stream URLs are preserved for downstream validation, while other link formats
are retained in the discovery output for inspection and logging.

> **In the full streaming pipeline** (`run_pipeline.py`), `DirectoryAgent` runs
> concurrently with `SearchAgent` using `asyncio.gather`. Both emit candidates
> via their `.stream()` async generator into a shared `asyncio.Queue`, but only
> `.m3u8` URLs are forwarded to `ValidationAgent`; non-HLS candidates are logged
> to `candidates/discovery_non_hls.jsonl`.
> Running this cell on its own executes `DirectoryAgent` in isolation.

### Crawler behaviour

- **`DirectoryTraversalSkill`** — follows sub-category links up to `max_depth=5` to reach
  individual camera pages (e.g. `/en/usa/new-york/niagara.html` is 4 segments deep)
- **`FeedExtractionSkill`** — scans static HTML for direct stream links and other embedded camera references
- **No discovery prefiltering by link type** — discovery keeps `.m3u8`, MJPEG, embed pages, and other camera URLs so operators can inspect all discovered formats
- **`DirectoryAgent.stream()`** — yields candidates batch-by-batch (`BATCH_SIZE=5` sources
  at a time) so the pipeline can validate early results while later sources are still being crawled
- **robots.txt checks run concurrently** — all source domains are checked in parallel
  before crawling begins, so blocked domains add no serial latency
- `MAX_PAGES_PER_SOURCE = 100` guard prevents runaway crawls


In [ ]:
!python scripts/run_discovery.py --tier 1 --output candidates/candidates.jsonl


## Step 3b — Run SearchAgent (DuckDuckGo discovery)

`SearchAgent` complements directory crawling with structured multi-language queries
against DuckDuckGo (no API key required). It generates queries for each Tier-N city
and keeps all public camera-result URLs that are not on the blocked-domain list.

Results are written to `candidates/search_candidates.jsonl`, then merged (deduplicated
by URL) with the directory-crawl results in Step 3c below.

> **In the full streaming pipeline** (`run_pipeline.py`), `SearchAgent` runs
> concurrently with `DirectoryAgent` via `asyncio.gather`. Its `.stream()` async
> generator emits candidates city-by-city directly into the shared `asyncio.Queue`,
> but only `.m3u8` URLs are forwarded to validation; the rest stay in discovery logs.
> Running this cell on its own executes `SearchAgent` in isolation.

### Search behaviour

- **`QueryGenerationSkill`** — generates query templates in English, Japanese, German,
  French, Spanish, Korean, Swedish, and Norwegian for each city
- **`SearchAgent.stream()`** — yields candidates city-by-city so the pipeline receives
  results incrementally rather than waiting for all cities to be searched
- **DuckDuckGo HTML endpoint** — no API key, polite 0.5 s delay between requests,
  concurrency capped at 5 simultaneous searches
- **`MAX_QUERIES_PER_CITY = 3`** guard avoids rate-limit triggers
- Only blocked domains (e.g. `insecam.org`, `shodan.io`) are filtered out before results are written


In [ ]:
!python scripts/run_search.py --tier 1 --output candidates/search_candidates.jsonl

## Step 3c — Merge discovery outputs and prepare validation input

Combines `candidates/candidates.jsonl` (DirectoryAgent) and `candidates/search_candidates.jsonl`
(SearchAgent) into a single deduplicated discovery file, then splits that merged set into:

- `candidates/candidates.jsonl` — **all** discovered candidate links for operator review
- `candidates/validation_candidates.jsonl` — only direct `.m3u8` links that should proceed to `ValidationAgent`
- `candidates/non_hls_candidates.jsonl` — non-`.m3u8` candidates logged at the discovery boundary

> **In the full streaming pipeline** (`run_pipeline.py`), cross-agent URL deduplication is handled inside
> `ValidationAgent.run_from_queue()`, and non-HLS candidates are logged to `candidates/discovery_non_hls.jsonl`
> instead of being forwarded into validation.


In [ ]:
import json, re
from pathlib import Path

_HLS_RE = re.compile(r"\.m3u8(\?|$)", re.IGNORECASE)

dir_path        = Path("candidates/candidates.jsonl")
search_path     = Path("candidates/search_candidates.jsonl")
merged_path     = Path("candidates/candidates.jsonl")
validation_path = Path("candidates/validation_candidates.jsonl")
non_hls_path    = Path("candidates/non_hls_candidates.jsonl")

dir_lines    = dir_path.read_text().splitlines()    if dir_path.exists()    else []
search_lines = search_path.read_text().splitlines() if search_path.exists() else []

seen_urls: set[str] = set()
merged_lines: list[str] = []
validation_lines: list[str] = []
non_hls_lines: list[str] = []

for line in dir_lines + search_lines:
    if not line.strip():
        continue
    try:
        payload = json.loads(line)
        url = payload.get("url", "")
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)
        merged_lines.append(json.dumps(payload, ensure_ascii=False))
        if url.lower().startswith(("http://", "https://")) and _HLS_RE.search(url):
            validation_lines.append(json.dumps(payload, ensure_ascii=False))
        else:
            non_hls_lines.append(json.dumps(payload, ensure_ascii=False))
    except json.JSONDecodeError:
        continue

merged_path.write_text("\n".join(merged_lines), encoding="utf-8")
validation_path.write_text("\n".join(validation_lines), encoding="utf-8")
non_hls_path.write_text("\n".join(non_hls_lines), encoding="utf-8")

print(f"Directory crawl          : {len(dir_lines)} candidates")
print(f"Search results           : {len(search_lines)} candidates")
print(f"Merged discovery total   : {len(merged_lines)} candidates → {merged_path}")
print(f"Validation (.m3u8 only)  : {len(validation_lines)} candidates → {validation_path}")
print(f"Logged non-.m3u8 links   : {len(non_hls_lines)} candidates → {non_hls_path}")


## Step 4 — Inspect discovery outputs

Preview candidate counts broken down by URL type. Discovery intentionally keeps **all** camera-link formats.
Only direct `.m3u8` HLS URLs are written to `validation_candidates.jsonl` and allowed to proceed into Step 5.

The standard HLS playlist extension is `.m3u8` — if you see `.h3u8`, treat that as a typo or malformed candidate rather than a valid stream type.

| Type | Meaning | Handling in Step 5 |
|------|---------|---------------------|
| `hls-stream`    | `.m3u8` HLS playlist URL             | ✅ Forwarded to validation |
| `html-page`     | Camera page URL (no `.m3u8` in URL)  | 📝 Logged at discovery only |
| `embed-page`    | Third-party player iframe URL        | 📝 Logged at discovery only |
| `youtube-embed` | YouTube nocookie embed URL           | 📝 Logged at discovery only |
| `mp4-file`      | Static MP4 video recording           | 📝 Logged at discovery only |
| `invalid-protocol` | Broken or protocol-relative URL    | 📝 Logged at discovery only unless normalization fixes it |


In [ ]:
import json, re
from pathlib import Path
from collections import Counter

all_path        = Path("candidates/candidates.jsonl")
validation_path = Path("candidates/validation_candidates.jsonl")
non_hls_path    = Path("candidates/non_hls_candidates.jsonl")

if not all_path.exists():
    print("candidates.jsonl not found - did the crawler run successfully?")
else:
    lines = all_path.read_text().splitlines()
    candidates = [json.loads(l) for l in lines if l.strip()]

    _hls_re     = re.compile(r"\.m3u8(\?|$)", re.IGNORECASE)
    _mp4_re     = re.compile(r"\.(mp4|webm|ogv)(\?|$)", re.IGNORECASE)
    _youtube_re = re.compile(r"youtube(?:-nocookie)?\.com/embed/", re.IGNORECASE)
    _embed_re   = re.compile(r"/embed[/\?]|embed\.", re.IGNORECASE)

    def url_type(url):
        if _hls_re.search(url):                                 return "hls-stream"
        if _mp4_re.search(url):                                 return "mp4-file"
        if _youtube_re.search(url):                             return "youtube-embed"
        if _embed_re.search(url):                               return "embed-page"
        if not url.lower().startswith(("http://","https://")): return "invalid-protocol"
        return "html-page"

    types   = Counter(url_type(c["url"]) for c in candidates)
    sources = Counter(c.get("source_directory", "unknown") for c in candidates)
    search_count = sum(n for s, n in sources.items() if s.startswith("search:"))
    crawl_count  = len(candidates) - search_count
    validation_count = sum(1 for _ in validation_path.read_text().splitlines() if _.strip()) if validation_path.exists() else 0
    non_hls_count = sum(1 for _ in non_hls_path.read_text().splitlines() if _.strip()) if non_hls_path.exists() else 0

    print(f"Total discovered candidates : {len(candidates)}")
    print(f"  From directory crawl     : {crawl_count}")
    print(f"  From search results      : {search_count}")
    print(f"  Forwarded to validation  : {validation_count}")
    print(f"  Logged at discovery only : {non_hls_count}")
    print()
    print("-- URL type breakdown --")
    print(f"  hls-stream       : {types['hls-stream']:>4}  ✅ forwarded to validation")
    print(f"  html-page        : {types['html-page']:>4}  📝 logged only")
    print(f"  embed-page       : {types['embed-page']:>4}  📝 logged only")
    print(f"  youtube-embed    : {types['youtube-embed']:>4}  📝 logged only")
    print(f"  mp4-file         : {types['mp4-file']:>4}  📝 logged only")
    print(f"  invalid-protocol : {types['invalid-protocol']:>4}  📝 logged unless normalized upstream")
    print()
    print("-- Top source directories --")
    for src, n in sources.most_common(15):
        print(f"  {n:>4}  {src}")
    print()
    print("-- First 10 discovery candidates --")
    for c in candidates[:10]:
        t = url_type(c["url"])
        mark = "✅" if t == "hls-stream" else "📝"
        print(f"  {mark} [{t:<16}]  {c['url']}")
        print(f"               label={c.get('label','-')}  city={c.get('city','-')}, {c.get('country','-')}")
        print()


## Step 5 — Validate `.m3u8` candidates only

`ValidationAgent` deep-probes only the direct `.m3u8` URLs prepared in `validation_candidates.jsonl`
and uses **ffprobe** plus **ffmpeg** frame sampling to confirm the stream is genuinely active
(real video frames present).

### Discovery-to-validation boundary

- Discovery keeps all candidate link formats for operator review
- Only direct `.m3u8` links are forwarded into Step 5
- Non-HLS candidates remain logged in `non_hls_candidates.jsonl` and do not go further for now

| URL type | Outcome |
|----------|---------|
| `.m3u8` with valid `#EXTM3U` | ✅ Probed; status set by ffprobe/ffmpeg |
| `.m3u8` missing protocol (`//…`) | ✅ Normalized during extraction when possible; otherwise ❌ excluded from validation input |
| HTML page / embed page | 📝 Logged at discovery only |
| YouTube / MP4 / MJPEG | 📝 Logged at discovery only |
| `.h3u8` | 📝 Invalid HLS extension; treat as malformed discovery output |

### ffprobe frame-level stream status (`use_ffprobe_validation=True`, default)

After HTTP confirmation that the `.m3u8` playlist is reachable, **ffprobe** inspects the stream and
**ffmpeg** decodes a short sample of frames for image analysis.


In [ ]:
!python scripts/run_validation.py --input candidates/validation_candidates.jsonl --output candidates/validated.jsonl


## Step 6 — Inspect validated cameras

Pipeline funnel, stream-status breakdown, and all validated HLS stream records.
Every retained URL in the output is a directly playable `.m3u8` stream — no user interaction required.

Stream status is set by **ffprobe/ffmpeg** frame analysis (when `WCD_USE_FFPROBE_VALIDATION=true`,
the default). Each record carries one of three statuses:

| `status`  | Meaning |
|-----------|---------|
| `live`    | ffprobe decoded frames with real content and motion detected |
| `unknown` | ffprobe decoded frames but they were blank or frozen, OR HTTP probe was inconclusive |
| `dead`    | ffprobe could not fetch segments, OR the stream failed definitively |

All three statuses are preserved in `validated.jsonl`. Dead streams should stay in the review queue unless they are explicitly re-validated and approved for removal.

> **Codebase alignment:** the runtime is now strictly **HLS-only**. Validation rejects non-`.m3u8` URLs before probing, so any non-HLS discovery candidates remain in discovery logs and never appear in `validated.jsonl`.


In [ ]:
import json, re
from pathlib import Path
from collections import Counter

validated_path  = Path("candidates/validated.jsonl")
candidates_path = Path("candidates/candidates.jsonl")

if not validated_path.exists():
    print("validated.jsonl not found - did validation run successfully?")
else:
    records = [json.loads(l) for l in validated_path.read_text().splitlines() if l.strip()]
    n_cands = sum(1 for l in candidates_path.read_text().splitlines() if l.strip()) \
              if candidates_path.exists() else "?"

    located   = [r for r in records if r.get("latitude") is not None]
    unlocated = [r for r in records if r.get("latitude") is None]

    _hls_re = re.compile(r"\.m3u8(\?|$)", re.IGNORECASE)

    def stream_category(r):
        url = r.get("url", "")
        ft  = r.get("feed_type", "")
        if _hls_re.search(url) and ft in ("HLS_master", "HLS_stream"):
            return f"hls-stream ✅ [{ft}]"
        return f"unexpected-non-hls ❌ [{ft}]"

    categories = Counter(stream_category(r) for r in records)
    valid_count   = sum(n for k, n in categories.items() if "✅" in k)
    invalid_count = sum(n for k, n in categories.items() if "❌" in k)
    status_counts = Counter(r.get("status", "unknown") for r in records)

    print("-- Pipeline funnel ------------------------------------------")
    print(f"  Candidates (discovery + search) : {n_cands}")
    print(f"  Validated records               : {len(records)}")
    print(f"    with coordinates              : {len(located)}")
    print(f"    without coords (included)     : {len(unlocated)}")
    print(f"    active HLS streams            : {valid_count}")
    print(f"    unexpected non-HLS records    : {invalid_count}")
    print()
    print("-- Status breakdown ----------------------------------------")
    for status, n in status_counts.items():
        print(f"  {status:<8} : {n}")
    print()
    print("-- Feed-type sanity check ----------------------------------")
    for cat, n in categories.items():
        print(f"  {cat:<32} : {n}")
    print()
    print("-- First 10 validated records ------------------------------")
    for r in records[:10]:
        coords = (f"{r['latitude']:.4f},{r['longitude']:.4f}"
                  if r.get('latitude') is not None and r.get('longitude') is not None
                  else "no-coords")
        print(f"- {r['status']:<7} {r['feed_type']:<10} {coords:<22} {r['label']}\n  {r['url']}")


## Step 7 — Build catalog (CatalogAgent)

`CatalogAgent` takes the validated records and:
1. **Deduplicates** — merges records with identical or near-identical stream URLs
2. **Exports** `camera.geojson` — GeoJSON FeatureCollection (RFC 7946 `[lon, lat]`)
3. **Exports** `cameras.md` — human-readable camera link list
4. **Snapshots** — writes a dated copy to `snapshots/camera_YYYY-MM-DD.geojson`

Both output files are written to the project root so that `map.html` can auto-load `camera.geojson`.

In [ ]:
!python scripts/run_catalog.py --input candidates/validated.jsonl --output .

## Step 8 — Inspect catalog outputs

Summary of `camera.geojson` features, coordinate coverage, and a preview of `cameras.md`.
Serve `map.html` locally to view the interactive map (camera.geojson must be co-located).

In [ ]:
import json
from pathlib import Path
from collections import Counter

geojson_path = Path("camera.geojson")
md_path      = Path("cameras.md")

if not geojson_path.exists():
    print("camera.geojson not found — did the catalog step run successfully?")
else:
    gj = json.loads(geojson_path.read_text())
    features = gj.get("features", [])

    located   = [f for f in features if f["geometry"] is not None]
    unlocated = [f for f in features if f["geometry"] is None]
    countries = Counter(
        f["properties"].get("country", "unknown") for f in features
    )
    feed_types = Counter(
        f["properties"].get("feed_type", "unknown") for f in features
    )
    sources = Counter(
        f["properties"].get("source_directory", "unknown") for f in features
    )

    print("── camera.geojson ───────────────────────────────────────")
    print(f"  Total features          : {len(features)}")
    print(f"  With coordinates        : {len(located)}")
    print(f"  Without coordinates     : {len(unlocated)}")
    print()
    print("── Feed types ───────────────────────────────────────────")
    for ft, n in feed_types.most_common():
        print(f"  {ft:<20} : {n}")
    print()
    print("── Top countries ────────────────────────────────────────")
    for country, n in countries.most_common(15):
        print(f"  {country:<30} : {n}")
    print()
    print("── Top source directories ───────────────────────────────")
    for src, n in sources.most_common(10):
        print(f"  {n:>4}  {src}")
    print()
    print("── First 5 features ─────────────────────────────────────")
    for f in features[:5]:
        p = f["properties"]
        geo = f["geometry"]
        coords = f"{geo['coordinates'][1]:.3f},{geo['coordinates'][0]:.3f}" if geo else "no-coords"
        print(f"  [{p.get('feed_type','?'):<14}] [{coords}]")
        print(f"    {p.get('label','-')} — {p.get('city','-')}, {p.get('country','-')}")
        print(f"    {p.get('url','')}")
        print()

    if md_path.exists():
        lines = md_path.read_text().splitlines()
        print(f"── cameras.md ({len(lines)} lines) — first 20 ──────────────────")
        print("\n".join(lines[:20]))

    print()
    print("── To view the interactive map ──────────────────────────")
    print("  python -m http.server 8000")
    print("  open http://localhost:8000/map.html")

## Step 9 — Export intermediate datasets as GeoJSON

Three supplementary GeoJSON exports for debugging and cross-pipeline comparison.
All three reuse `DeduplicationSkill` and `GeoJSONExportSkill` from the catalog layer.

| Output file | Source | Filter | Notes |
|-------------|--------|--------|-------|
| `validated.geojson` | `candidates/validated.jsonl` | all validated records | Deduped `CameraRecord` objects with full geo-enrichment |
| `candidates.geojson` | `candidates/candidates.jsonl` | HLS `.m3u8` only | Raw discovery candidates geo-enriched on the fly |
| `search_candidates.geojson` | `candidates/search_candidates.jsonl` | HLS `.m3u8` only | Search candidates geo-enriched on the fly |

**Why filter to HLS-only?** The production pipeline is hardcoded to accept only direct `.m3u8` URLs.
Filtering these debug GeoJSON exports the same way keeps notebook artifacts aligned with `ValidationAgent` and `CatalogAgent`.

> **Cells 9b and 9c use `await`** — this requires IPython ≥ 7 (Jupyter Lab, Jupyter Notebook 6+,
> Google Colab) which enables top-level `await` inside notebook cells.


In [ ]:
"""
Step 9a — Export validated.geojson.

Loads candidates/validated.jsonl (full CameraRecord objects produced by
ValidationAgent), deduplicates using the same DeduplicationSkill that
CatalogAgent uses, then exports to validated.geojson via GeoJSONExportSkill.

This gives a GeoJSON snapshot of every record that passed validation before
the final catalog deduplication step.
"""
import json
from pathlib import Path
from webcam_discovery.schemas import CameraRecord
from webcam_discovery.skills.catalog import (
    DeduplicationSkill, DeduplicationInput,
    GeoJSONExportSkill, GeoJSONExportInput,
)

input_path  = Path("candidates/validated.jsonl")
output_path = Path("validated.geojson")

if not input_path.exists():
    print(f"{input_path} not found — run Step 5 first")
else:
    records = [
        CameraRecord(**json.loads(line))
        for line in input_path.read_text().splitlines()
        if line.strip()
    ]
    print(f"Loaded {len(records)} validated records")

    # Deduplicate using the same skill CatalogAgent uses
    dedup_skill = DeduplicationSkill()
    catalog: list[CameraRecord] = []
    for record in records:
        result = dedup_skill.run(DeduplicationInput(
            candidate_record=record,
            existing_catalog=catalog,
        ))
        if result.is_duplicate:
            if result.merged_record and result.canonical_record:
                idx = next(
                    (i for i, r in enumerate(catalog) if r.id == result.canonical_record.id),
                    None,
                )
                if idx is not None:
                    catalog[idx] = result.merged_record
        else:
            catalog.append(record)

    print(f"After dedup: {len(catalog)} records ({len(records) - len(catalog)} duplicates removed)")

    result = GeoJSONExportSkill().run(GeoJSONExportInput(cameras=catalog, output_path=output_path))
    print(f"Exported : {result.exported} features → {output_path}")
    print(f"Skipped  : {result.skipped} records (no coordinates)")

In [ ]:
"""
Step 9b — Export candidates.geojson (HLS .m3u8 only).

Filters candidates.jsonl to direct HLS (.m3u8) stream URLs — those already
resolved to a live stream URL by FeedExtractionSkill during directory crawling.
Non-HLS URLs (HTML pages, embed pages, MJPEG, MP4) are excluded here; they
would be dropped anyway by the hls_only filter in Step 5.

Geocoding uses GeoEnrichmentSkill, which calls the Ollama LLM by default
(USE_LLM_GEODECODE=True).  Each camera's location string (label, city, country)
is sent to the configured Ollama model and returns location, latitude, longitude,
country, region, and continent.  Results are cached process-wide.
"""
import asyncio, json, re
from pathlib import Path
from slugify import slugify
from webcam_discovery.schemas import CameraCandidate, CameraRecord
from webcam_discovery.skills.catalog import (
    GeoEnrichmentSkill, GeoEnrichmentInput,
    GeoJSONExportSkill, GeoJSONExportInput,
)

_HLS_RE = re.compile(r'\.m3u8(\?|$)', re.IGNORECASE)

input_path  = Path("candidates/candidates.jsonl")
output_path = Path("candidates.geojson")

if not input_path.exists():
    print(f"{input_path} not found — run Step 3a first")
else:
    all_cands = [
        CameraCandidate(**json.loads(line))
        for line in input_path.read_text().splitlines()
        if line.strip()
    ]

    hls_cands = [
        c for c in all_cands
        if _HLS_RE.search(c.url) and c.url.lower().startswith(("http://", "https://"))
    ]
    print(f"candidates.jsonl : {len(all_cands)} total → {len(hls_cands)} HLS (.m3u8) streams")

    geo_skill = GeoEnrichmentSkill()

    async def _enrich_all(cands):
        tasks = [
            geo_skill.run(GeoEnrichmentInput(
                city    = c.city    if c.city    and c.city    != "Unknown" else None,
                country = c.country if c.country and c.country != "Unknown" else None,
                label   = c.label   if c.label   and c.label   != c.city    else None,
                url     = c.url,
            ))
            for c in cands
        ]
        return await asyncio.gather(*tasks, return_exceptions=True)

    geo_results = await _enrich_all(hls_cands)

    records: list[CameraRecord] = []
    for candidate, geo in zip(hls_cands, geo_results):
        city    = candidate.city    or "Unknown"
        country = candidate.country or "Unknown"
        label   = candidate.label   or city
        lat = lon = continent = region = None
        if isinstance(geo, Exception):
            pass
        elif geo is not None:
            lat, lon  = geo.latitude, geo.longitude
            continent = geo.continent or "Unknown"
            region    = geo.region
            if geo.country:
                country = geo.country

        records.append(CameraRecord(
            id               = slugify(f"{city} {label}", max_length=80, word_boundary=True, separator="-") or "camera",
            label            = label,
            city             = city,
            country          = country,
            continent        = continent or "Unknown",
            region           = region,
            latitude         = lat,
            longitude        = lon,
            url              = candidate.url,
            feed_type        = "HLS_stream",
            source_directory = candidate.source_directory,
            source_refs      = candidate.source_refs or [candidate.url],
            legitimacy_score = "low",
            status           = "unknown",
            notes            = candidate.notes,
        ))

    located = sum(1 for r in records if r.latitude is not None)
    print(f"Geo-enriched: {located}/{len(records)} records have coordinates")

    result = GeoJSONExportSkill().run(GeoJSONExportInput(cameras=records, output_path=output_path))
    print(f"Exported : {result.exported} features → {output_path}")
    print(f"Skipped  : {result.skipped} records (no coordinates)")


In [ ]:
"""
Step 9c — Export search_candidates.geojson (HLS .m3u8 only).

Filters search_candidates.jsonl to direct HLS (.m3u8) stream URLs,
geo-enriches each record using GeoEnrichmentSkill, then exports via
GeoJSONExportSkill.

Geocoding uses the Ollama LLM by default (USE_LLM_GEODECODE=True): each
camera's location string (label, city, country) is sent to the configured
Ollama model and returns location, latitude, longitude, country, region,
and continent.  Results are cached process-wide.

Note: SearchAgent returns DuckDuckGo result pages, so most URLs are HTML
pages rather than direct stream links.  HLS hits here are rare but valuable.
"""
import asyncio, json, re
from pathlib import Path
from slugify import slugify
from webcam_discovery.schemas import CameraCandidate, CameraRecord
from webcam_discovery.skills.catalog import (
    GeoEnrichmentSkill, GeoEnrichmentInput,
    GeoJSONExportSkill, GeoJSONExportInput,
)

_HLS_RE = re.compile(r'\.m3u8(\?|$)', re.IGNORECASE)

input_path  = Path("candidates/search_candidates.jsonl")
output_path = Path("search_candidates.geojson")

if not input_path.exists():
    print(f"{input_path} not found — run Step 3b first")
else:
    all_cands = [
        CameraCandidate(**json.loads(line))
        for line in input_path.read_text().splitlines()
        if line.strip()
    ]

    hls_cands = [
        c for c in all_cands
        if _HLS_RE.search(c.url) and c.url.lower().startswith(("http://", "https://"))
    ]
    print(f"search_candidates.jsonl : {len(all_cands)} total → {len(hls_cands)} HLS (.m3u8) streams")

    geo_skill = GeoEnrichmentSkill()

    async def _enrich_all(cands):
        tasks = [
            geo_skill.run(GeoEnrichmentInput(
                city    = c.city    if c.city    and c.city    != "Unknown" else None,
                country = c.country if c.country and c.country != "Unknown" else None,
                label   = c.label   if c.label   and c.label   != c.city    else None,
                url     = c.url,
            ))
            for c in cands
        ]
        return await asyncio.gather(*tasks, return_exceptions=True)

    geo_results = await _enrich_all(hls_cands)

    records: list[CameraRecord] = []
    for candidate, geo in zip(hls_cands, geo_results):
        city    = candidate.city    or "Unknown"
        country = candidate.country or "Unknown"
        label   = candidate.label   or city
        lat = lon = continent = region = None
        if isinstance(geo, Exception):
            pass
        elif geo is not None:
            lat, lon  = geo.latitude, geo.longitude
            continent = geo.continent or "Unknown"
            region    = geo.region
            if geo.country:
                country = geo.country

        records.append(CameraRecord(
            id               = slugify(f"{city} {label}", max_length=80, word_boundary=True, separator="-") or "camera",
            label            = label,
            city             = city,
            country          = country,
            continent        = continent or "Unknown",
            region           = region,
            latitude         = lat,
            longitude        = lon,
            url              = candidate.url,
            feed_type        = "HLS_stream",
            source_directory = candidate.source_directory,
            source_refs      = candidate.source_refs or [candidate.url],
            legitimacy_score = "low",
            status           = "unknown",
            notes            = candidate.notes,
        ))

    located = sum(1 for r in records if r.latitude is not None)
    print(f"Geo-enriched: {located}/{len(records)} records have coordinates")

    result = GeoJSONExportSkill().run(GeoJSONExportInput(cameras=records, output_path=output_path))
    print(f"Exported : {result.exported} features → {output_path}")
    print(f"Skipped  : {result.skipped} records (no coordinates)")
